# PRA Dataset Explorer

Use this notebook to compare sample datasets, inspect their reference tables, and verify that each stage can feed the same collator and DataLoader path.

In [1]:
from pathlib import Path
import sys

repo = Path.cwd()
if repo.name == "nb":
    repo = repo.parent
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo))

from data.datamodules import PRADataModule
from data.datasets import DATASET_REGISTRY
from data.tokenizer import PRATokenizer

## Available Dataset Stages

In [2]:
sorted(DATASET_REGISTRY)

['stage0_synthetic_memory',
 'stage1_hierarchical_synthetic',
 'stage2_code_repos',
 'stage3_wikipedia',
 'stage4_books',
 'stage5_technical_docs',
 'stage6_github_repos']

## Load Every Stage And Summarize Shape

In [3]:
dataset_summaries = []
datasets = {}

for stage, dataset_cls in sorted(DATASET_REGISTRY.items()):
    ds = dataset_cls(repo / "data", max_examples=3)
    datasets[stage] = ds
    first = ds[0] if len(ds) else None
    dataset_summaries.append(
        {
            "stage": stage,
            "dataset_name": ds.dataset_name,
            "samples": len(ds),
            "references": len(ds.reference_rows),
            "first_question": first.question if first else None,
            "first_targets": first.target_reference_ids if first else [],
        }
    )

dataset_summaries

[{'stage': 'stage0_synthetic_memory',
  'dataset_name': 'synthetic_memory_qa',
  'samples': 3,
  'references': 4,
  'first_question': 'Which animal hunts mice? <REF_1> <REF_2> <REF_3>',
  'first_targets': [1]},
 {'stage': 'stage1_hierarchical_synthetic',
  'dataset_name': 'hierarchical_reference',
  'samples': 3,
  'references': 3,
  'first_question': 'What is the JWT expiration? <REF_1> <REF_2>',
  'first_targets': [1]},
 {'stage': 'stage2_code_repos',
  'dataset_name': 'code_repository',
  'samples': 2,
  'references': 1,
  'first_question': 'Which function issues the login JWT? <REF_1>',
  'first_targets': [1]},
 {'stage': 'stage3_wikipedia',
  'dataset_name': 'wikipedia',
  'samples': 1,
  'references': 1,
  'first_question': 'Who discovered penicillin? <REF_1>',
  'first_targets': [1]},
 {'stage': 'stage4_books',
  'dataset_name': 'books',
  'samples': 1,
  'references': 1,
  'first_question': 'What does Alice follow? <REF_1>',
  'first_targets': [1]},
 {'stage': 'stage5_technical

## Inspect One Stage In Detail

In [4]:
stage = "stage0_synthetic_memory"
ds = datasets[stage]
sample = ds[0]
table = ds.build_reference_table(sample)

{
    "sample_id": sample.id,
    "question": sample.question,
    "answer": sample.answer,
    "target_reference_ids": sample.target_reference_ids,
    "handles": [handle for handle in table.all()],
}

{'sample_id': 'q0_1',
 'question': 'Which animal hunts mice? <REF_1> <REF_2> <REF_3>',
 'answer': 'cat',
 'target_reference_ids': [1],
 'handles': [ReferenceHandle(id=1, token='<REF_1>', uri='memory://animal/cat', summary='Facts about cat.', metadata={'stage': 0, 'text': 'Title: Animal 1. The animal is cat. It has whiskers. It hunts mice. Secret code: CODE17.', 'title': 'Animal 1', 'anchors': []}),
  ReferenceHandle(id=2, token='<REF_2>', uri='memory://animal/dog', summary='Facts about dog.', metadata={'stage': 0, 'text': 'Title: Animal 2. The animal is dog. It has bark. It guards houses. Secret code: CODE34.', 'title': 'Animal 2', 'anchors': []}),
  ReferenceHandle(id=3, token='<REF_3>', uri='memory://animal/owl', summary='Facts about owl.', metadata={'stage': 0, 'text': 'Title: Animal 3. The animal is owl. It has night vision. It hunts at night. Secret code: CODE51.', 'title': 'Animal 3', 'anchors': []}),
  ReferenceHandle(id=4, token='<REF_4>', uri='memory://animal/bee', summary='Fa

## Compare Tokenizer Behavior Across Reference Tokens

In [5]:
corpus = []
for ds in datasets.values():
    for sample in ds:
        corpus.append(sample.question + " " + sample.answer)
        for ref in sample.references:
            corpus.append(ref.summary or "")
            corpus.append(str(ref.metadata.get("text", "")))

tokenizer = PRATokenizer(corpus)
probe = "Check <REF_1>, <REF_2>, and <REF_99>."
ids = tokenizer.encode(probe)

{
    "probe": probe,
    "ids": ids,
    "decoded": tokenizer.decode(ids),
    "reference_vocab": {k: v for k, v in tokenizer.stoi.items() if k.startswith("<REF_")},
}

{'probe': 'Check <REF_1>, <REF_2>, and <REF_99>.',
 'ids': [40,
  77,
  74,
  72,
  80,
  5,
  101,
  17,
  5,
  102,
  17,
  5,
  70,
  83,
  73,
  5,
  105,
  19],
 'decoded': 'Check <REF_1>, <REF_2>, and <REF_99>.',
 'reference_vocab': {'<REF_0>': 100,
  '<REF_1>': 101,
  '<REF_2>': 102,
  '<REF_3>': 103,
  '<REF_4>': 104,
  '<REF_99>': 105}}

## Verify Every Stage Can Build A DataLoader Batch

In [6]:
loader_summaries = []

for stage in sorted(DATASET_REGISTRY):
    dm = PRADataModule(
        dataset_stage=stage,
        data_dir=repo / "data",
        max_examples=4,
        batch_size=2,
        max_seq_len=80,
        shuffle=False,
    ).load()
    batch = next(iter(dm.train_loader()))
    loader_summaries.append(
        {
            "stage": stage,
            "input_ids": tuple(batch["input_ids"].shape),
            "labels": tuple(batch["labels"].shape),
            "attention_mask": tuple(batch["attention_mask"].shape),
            "tables": len(batch["reference_tables"]),
            "metadata_ids": [item["id"] for item in batch["metadata"]],
        }
    )

loader_summaries

[{'stage': 'stage0_synthetic_memory',
  'input_ids': (1, 34),
  'labels': (1, 34),
  'attention_mask': (1, 34),
  'tables': 1,
  'metadata_ids': ['q0_2']},
 {'stage': 'stage1_hierarchical_synthetic',
  'input_ids': (1, 42),
  'labels': (1, 42),
  'attention_mask': (1, 42),
  'tables': 1,
  'metadata_ids': ['q1_2']},
 {'stage': 'stage2_code_repos',
  'input_ids': (2, 57),
  'labels': (2, 57),
  'attention_mask': (2, 57),
  'tables': 2,
  'metadata_ids': ['q2_1', 'q2_2']},
 {'stage': 'stage3_wikipedia',
  'input_ids': (1, 47),
  'labels': (1, 47),
  'attention_mask': (1, 47),
  'tables': 1,
  'metadata_ids': ['q3_1']},
 {'stage': 'stage4_books',
  'input_ids': (1, 39),
  'labels': (1, 39),
  'attention_mask': (1, 39),
  'tables': 1,
  'metadata_ids': ['q4_1']},
 {'stage': 'stage5_technical_docs',
  'input_ids': (2, 80),
  'labels': (2, 80),
  'attention_mask': (2, 80),
  'tables': 2,
  'metadata_ids': ['q5_1', 'q5_2']},
 {'stage': 'stage6_github_repos',
  'input_ids': (2, 74),
  'labels'

## Choose A Stage For Experiments

In [ ]:
experiment_stage = "stage1_hierarchical_synthetic"
experiment_dm = PRADataModule(
    dataset_stage=experiment_stage,
    data_dir=repo / "data",
    max_examples=8,
    batch_size=2,
    max_seq_len=96,
    shuffle=True,
).load()

experiment_batch = next(iter(experiment_dm.train_loader()))
{
    "stage": experiment_stage,
    "tokenizer_vocab_size": experiment_dm.tokenizer.vocab_size,
    "input_preview": experiment_dm.tokenizer.decode(experiment_batch["input_ids"][0].tolist()).rstrip(),
    "target_answer": experiment_batch["metadata"][0]["answer"],
    "available_refs": [ref.uri for ref in experiment_batch["metadata"][0]["references"]],
}